# 📚 CBSE Class XII Learning Buddy Bot
### Full RAG Pipeline — Reads from your uploaded ZIP

| Step | What happens |
|------|--------------|
| 1 | Install dependencies |
| 2 | Set API keys |
| 3 | Upload & unzip your Class12 books |
| 4 | Scan chapters + build manifest |
| 5 | Extract text + images |
| 6 | Chunk content |
| 7 | Embed with OpenAI |
| 8 | Upload to Pinecone |
| 9 | Test retrieval |
| 10 | RAG Q&A |
| 11 | Gradio chatbot UI |

> 💡 **Colab tip:** Runtime → Change runtime type → T4 GPU

## 🔧 Step 1 — Install Dependencies

In [1]:
!uv pip install pymupdf pinecone openai gradio tqdm --quiet

## 🔑 Step 2 — API Keys
> [OpenAI keys](https://platform.openai.com/api-keys) · [Pinecone keys](https://app.pinecone.io)

In [2]:
OPENAI_API_KEY   = "sk-proj- "      # ← your OpenAI key
PINECONE_API_KEY = "pcsk_42W6br _"    # ← your Pinecone key


# ── Pinecone ─────────────────────────────────
PINECONE_INDEX  = "cbse-class12-from-pc2"
PINECONE_CLOUD  = "aws"
PINECONE_REGION = "us-east-1"

# ── Models ───────────────────────────────────
EMBED_MODEL = "text-embedding-ada-002"
LLM_MODEL   = "gpt-4o"

# ── Chunking ─────────────────────────────────
CHUNK_SIZE    = 1000
CHUNK_OVERLAP = 200
TOP_K         = 8

print("✅ Config ready")

✅ Config ready


## 📦 Step 3 — Upload & Unzip Your Class12 Books

Upload your `Class12.zip` file when prompted.

Expected zip structure:
```
Class12/
├── Biology/
│   ├── chapter_01.pdf
│   ├── chapter_02.pdf
│   └── prelims.pdf      ← auto-skipped
├── Chemistry_Part1/
│   ├── chapter_01.pdf
│   ├── answers.pdf      ← auto-skipped
│   └── lech1a1.pdf      ← auto-skipped
└── ...
```

In [3]:
ls -alh ./

total 507M
drwxrwxr-x   6 bala bala 4.0K Mar 29 15:42 ./
drwxrwxr-x  11 bala bala 4.0K Mar 29 01:50 ../
-rw-rw-r--   1 bala bala  77K Mar 29 15:39 cbse_buddy_v3.ipynb
-rw-rw-r--   1 bala bala  88K Mar 29 15:42 cbse_buddy_v4.ipynb
drwxrwxr-x  20 bala bala 4.0K Mar 29 15:33 Class12/
-rw-rw-r--   1 bala bala 507M Mar 29 01:30 Class12.zip
drwxrwxr-x 140 bala bala  12K Mar 29 15:36 images/
drwxrwxr-x   2 bala bala 4.0K Mar 29 15:41 .ipynb_checkpoints/
drwxrwxr-x   6 bala bala 4.0K Mar 29 01:55 .venv/


In [5]:
import zipfile, os
# from google.colab import files

# print("📤 Select your Class12.zip to upload...")
# uploaded = files.upload()

zip_filename = "./Class12.zip"
print(f"\n📦 Unzipping: {zip_filename}")

# with zipfile.ZipFile(zip_filename, "r") as zf:
#     zf.extractall("./")

# Auto-detect the Class12 root folder
for candidate in ["./Class12", f"./{zip_filename.replace('.zip', '')}"]:
    if os.path.isdir(candidate):
        CLASS12_DIR = candidate
        break
else:
    # Fallback: find the directory that contains subject folders
    for root, dirs, _ in os.walk("/content"):
        if "Biology" in dirs or "Chemistry_Part1" in dirs:
            CLASS12_DIR = root
            break

print(f"\n📂 Class12 root: {CLASS12_DIR}")
subjects = [d for d in sorted(os.listdir(CLASS12_DIR))
            if os.path.isdir(os.path.join(CLASS12_DIR, d))]
print(f"✅ Found {len(subjects)} subjects: {subjects}")


📦 Unzipping: ./Class12.zip

📂 Class12 root: ./Class12
✅ Found 18 subjects: ['Accountancy_Part1', 'Accountancy_Part2', 'Biology', 'Biotechnology', 'Business_Studies_Part1', 'Business_Studies_Part2', 'Chemistry_Part1', 'Chemistry_Part2', 'Computer_Science', 'English_Flamingo', 'Informatics_Practices', 'Maths_Part1', 'Maths_Part2', 'Physics_Part1', 'Physics_Part2', 'Political_Science_Part1', 'Political_Science_Part2', 'Psychology']


## 🗂️ Step 4 — Scan Chapters & Build Manifest

Automatically:
- Finds all `chapter_XX.pdf` files
- Skips `prelims.pdf`, `answers.pdf`, `glossary.pdf`, and oddly-named files
- Maps each chapter to its correct title and Pinecone namespace

In [8]:
import re

# ── Files to always skip ──────────────────────────────────────
SKIP_FILES = {"prelims.pdf", "answers.pdf", "glossary.pdf"}

def is_valid_chapter(filename):
    """Only process files named chapter_XX.pdf"""
    return filename.startswith("chapter_") and filename.endswith(".pdf")

def get_chapter_num(filename):
    m = re.search(r"chapter_(\d+)", filename)
    return int(m.group(1)) if m else 0

# ── Subject → Pinecone namespace ─────────────────────────────
NAMESPACE_MAP = {
    "Accountancy_Part1":        "accountancy",
    "Accountancy_Part2":        "accountancy",
    "Biology":                  "biology",
    "Biotechnology":            "biotechnology",
    "Business_Studies_Part1":   "business-studies",
    "Business_Studies_Part2":   "business-studies",
    "Chemistry_Part1":          "chemistry",
    "Chemistry_Part2":          "chemistry",
    "Computer_Science":         "computer-science",
    "English_Flamingo":         "english",
    "Informatics_Practices":    "informatics-practices",
    "Maths_Part1":              "mathematics",
    "Maths_Part2":              "mathematics",
    "Physics_Part1":            "physics",
    "Physics_Part2":            "physics",
    "Political_Science_Part1":  "political-science",
    "Political_Science_Part2":  "political-science",
    "Psychology":               "psychology",
}

# ── Chapter titles per subject ────────────────────────────────
# These are the actual chapter titles from the NCERT books
# (chapter number = position in the PDF folder, NOT NCERT URL number)
CHAPTER_TITLES = {
    "Biology": {
        1:  "Sexual Reproduction in Flowering Plants",
        2:  "Human Reproduction",
        3:  "Reproductive Health",
        4:  "Principles of Inheritance and Variation",
        5:  "Molecular Basis of Inheritance",
        6:  "Evolution",
        7:  "Human Health and Disease",
        8:  "Microbes in Human Welfare",
        9:  "Biotechnology: Principles and Processes",
        10: "Biotechnology and its Applications",
        11: "Organisms and Populations",
        12: "Ecosystem",
        13: "Biodiversity and Conservation",
    },
    "Biotechnology": {
        1:  "Introduction to Biotechnology",
        2:  "Enzymes",
        3:  "Genes and Genomes",
        4:  "Genetic Engineering",
        5:  "Protein Structure and Engineering",
        6:  "Cell Culture",
        7:  "Microbial Biotechnology",
        8:  "Plant Biotechnology",
        9:  "Animal Biotechnology",
        10: "Medical Biotechnology",
        11: "Immunotechnology",
        12: "Environmental Biotechnology",
        13: "Bioinformatics",
    },
    "Chemistry_Part1": {
        1: "The Solid State",
        2: "Solutions",
        3: "Electrochemistry",
        4: "Chemical Kinetics",
        5: "Surface Chemistry",
    },
    "Chemistry_Part2": {
        1: "General Principles of Isolation of Elements",
        2: "The p-Block Elements",
        3: "The d and f Block Elements",
        4: "Coordination Compounds",
        5: "Haloalkanes and Haloarenes",
    },
    "Physics_Part1": {
        1: "Electric Charges and Fields",
        2: "Electrostatic Potential and Capacitance",
        3: "Current Electricity",
        4: "Moving Charges and Magnetism",
        5: "Magnetism and Matter",
        6: "Electromagnetic Induction",
        7: "Alternating Current",
        8: "Electromagnetic Waves",
    },
    "Physics_Part2": {
        1: "Ray Optics and Optical Instruments",
        2: "Wave Optics",
        3: "Dual Nature of Radiation and Matter",
        4: "Atoms",
        5: "Nuclei",
        6: "Semiconductor Electronics",
    },
    "Maths_Part1": {
        1: "Relations and Functions",
        2: "Inverse Trigonometric Functions",
        3: "Matrices",
        4: "Determinants",
        5: "Continuity and Differentiability",
        6: "Application of Derivatives",
    },
    "Maths_Part2": {
        1: "Integrals",
        2: "Application of Integrals",
        3: "Differential Equations",
        4: "Vector Algebra",
        5: "Three Dimensional Geometry",
        6: "Linear Programming",
        7: "Probability",
    },
    "Computer_Science": {
        1:  "Exception Handling",
        2:  "File Handling",
        3:  "Stack",
        4:  "Queue",
        5:  "Sorting",
        6:  "Searching",
        7:  "Understanding Data",
        8:  "Database Concepts",
        9:  "Structured Query Language",
        10: "Computer Networks",
        11: "Data Communication",
        12: "Security Aspects",
        13: "Societal Impacts",
    },
    "Informatics_Practices": {
        1: "Data Handling Using Pandas",
        2: "Data Visualization",
        3: "Database Query Using SQL",
        4: "More on SQL",
        5: "Societal Impacts",
        6: "Networking and the Internet",
        7: "Data Communication",
    },
    "Accountancy_Part1": {
        2: "Accounting for Partnership: Basic Concepts",
        3: "Reconstitution of a Partnership Firm — Admission of a Partner",
        4: "Reconstitution of a Partnership Firm — Retirement and Death",
    },
    "Accountancy_Part2": {
        1: "Accounting for Share Capital",
        2: "Issue and Redemption of Debentures",
        3: "Financial Statements of a Company",
        4: "Analysis of Financial Statements",
        5: "Accounting Ratios",
        6: "Cash Flow Statement",
    },
    "Business_Studies_Part1": {
        1: "Nature and Significance of Management",
        2: "Principles of Management",
        3: "Business Environment",
        4: "Planning",
        5: "Organising",
        6: "Staffing",
        7: "Directing",
        8: "Controlling",
    },
    "Business_Studies_Part2": {
        1: "Financial Management",
        2: "Financial Markets",
        3: "Marketing Management",
    },
    "English_Flamingo": {
        1:  "The Last Lesson",
        2:  "Lost Spring",
        3:  "Deep Water",
        4:  "The Rattrap",
        5:  "Indigo",
        6:  "Poets and Pancakes",
        7:  "The Interview",
        8:  "Going Places",
        11: "My Mother at Sixty-six (Poem)",
        12: "An Elementary School Classroom in a Slum (Poem)",
        13: "Keeping Quiet (Poem)",
        14: "A Thing of Beauty (Poem)",
        15: "A Roadside Stand (Poem)",
    },
    "Political_Science_Part1": {
        1: "The Cold War Era",
        2: "The End of Bipolarity",
        3: "US Hegemony in World Politics",
        4: "Alternative Centres of Power",
        5: "Contemporary South Asia",
        6: "International Organisations",
        7: "Security in the Contemporary World",
    },
    "Political_Science_Part2": {
        1: "Challenges of Nation Building",
        2: "Era of One-Party Dominance",
        3: "Politics of Planned Development",
        4: "India's External Relations",
        5: "Challenges to the Congress System",
        6: "Crisis of the Constitutional Order",
        7: "Rise of Popular Movements",
        8: "Recent Developments in Indian Politics",
    },
    "Psychology": {
        1: "Variations in Psychological Attributes",
        2: "Self and Personality",
        3: "Meeting Life Challenges",
        4: "Psychological Disorders",
        5: "Therapeutic Approaches",
        6: "Attitude and Social Cognition",
        7: "Social Influence and Group Processes",
    },
}

def get_chapter_title(subject, ch_num):
    return CHAPTER_TITLES.get(subject, {}).get(ch_num, f"Chapter {ch_num}")


# ── Build manifest ────────────────────────────────────────────
def scan_chapters(class12_dir, subject_filter=None):
    manifest = []
    for subject in sorted(os.listdir(class12_dir)):
        subj_dir = os.path.join(class12_dir, subject)
        if not os.path.isdir(subj_dir):
            continue
        if subject_filter and subject not in subject_filter:
            continue

        ns = NAMESPACE_MAP.get(subject, subject.lower().replace("_", "-"))

        for fname in sorted(os.listdir(subj_dir)):
            if fname in SKIP_FILES: continue
            if not is_valid_chapter(fname): continue

            ch_num  = get_chapter_num(fname)
            ch_title = get_chapter_title(subject, ch_num)
            pdf_path = os.path.join(subj_dir, fname)

            manifest.append({
                "subject":       subject,
                "namespace":     ns,
                "chapter_num":   ch_num,
                "chapter_title": ch_title,
                "pdf_path":      pdf_path,
            })
    return manifest


manifest = scan_chapters(CLASS12_DIR)

# Print manifest
cur = None
for ch in manifest:
    if ch["subject"] != cur:
        cur = ch["subject"]
        print(f"\n📚 {cur}  [{NAMESPACE_MAP.get(cur, '?')}]")
    print(f"   Ch{ch['chapter_num']:02d}  {ch['chapter_title']}")

print(f"\n✅ Total: {len(manifest)} chapters across {len(set(c['subject'] for c in manifest))} subjects")


📚 Accountancy_Part1  [accountancy]
   Ch02  Accounting for Partnership: Basic Concepts
   Ch03  Reconstitution of a Partnership Firm — Admission of a Partner
   Ch04  Reconstitution of a Partnership Firm — Retirement and Death

📚 Accountancy_Part2  [accountancy]
   Ch01  Accounting for Share Capital
   Ch02  Issue and Redemption of Debentures
   Ch03  Financial Statements of a Company
   Ch04  Analysis of Financial Statements
   Ch05  Accounting Ratios
   Ch06  Cash Flow Statement

📚 Biology  [biology]
   Ch01  Sexual Reproduction in Flowering Plants
   Ch02  Human Reproduction
   Ch03  Reproductive Health
   Ch04  Principles of Inheritance and Variation
   Ch05  Molecular Basis of Inheritance
   Ch06  Evolution
   Ch07  Human Health and Disease
   Ch08  Microbes in Human Welfare
   Ch09  Biotechnology: Principles and Processes
   Ch10  Biotechnology and its Applications
   Ch11  Organisms and Populations
   Ch12  Ecosystem
   Ch13  Biodiversity and Conservation

📚 Biotechnology  [bio

### 🎛️ Optional — Filter to specific subjects only
Useful to test with just 2-3 subjects before running everything.

In [9]:
# ── Option A: Run ALL subjects (comment this out to use Option B) ──
chapters_to_process = manifest

# ── Option B: Run specific subjects only ──
# chapters_to_process = scan_chapters(CLASS12_DIR, subject_filter=[
#     "Biology",
#     "Physics_Part1",
#     "Chemistry_Part1",
# ])

print(f"✅ Will process {len(chapters_to_process)} chapters")
for ns in sorted(set(c["namespace"] for c in chapters_to_process)):
    n = sum(1 for c in chapters_to_process if c["namespace"] == ns)
    print(f"   [{ns}] → {n} chapters")

✅ Will process 138 chapters
   [accountancy] → 9 chapters
   [biology] → 13 chapters
   [biotechnology] → 13 chapters
   [business-studies] → 11 chapters
   [chemistry] → 10 chapters
   [computer-science] → 13 chapters
   [english] → 13 chapters
   [informatics-practices] → 7 chapters
   [mathematics] → 13 chapters
   [physics] → 14 chapters
   [political-science] → 15 chapters
   [psychology] → 7 chapters


## 📄 Step 5 — Extract Text + Images from PDFs

In [10]:
import fitz  # PyMuPDF

os.makedirs("./images", exist_ok=True)

NOISE_HEADERS = {
    "BIOLOGY", "CHEMISTRY", "PHYSICS", "MATHEMATICS", "SCIENCE",
    "BIOTECHNOLOGY", "NCERT", "ACCOUNTANCY", "BUSINESS STUDIES",
    "ENGLISH", "PSYCHOLOGY", "POLITICAL SCIENCE",
    "INFORMATICS PRACTICES", "COMPUTER SCIENCE",
}

def clean_text(raw):
    kept = []
    for ln in raw.split("\n"):
        ln = ln.strip()
        if not ln or len(ln) < 3: continue
        if re.match(r"^\d+$", ln): continue
        if ln.upper() in NOISE_HEADERS: continue
        if re.match(r"^Reprint \d{4}", ln): continue
        kept.append(ln)
    return " ".join(kept).strip()

def is_real_image(w, h):
    if w <= 10 or h <= 10: return False        # 1px lines/borders
    if w * h < 5000: return False              # tiny icons
    if w > 900 and h < 250: return False       # repeating page banner
    return True

def extract_chapter(ch):
    """Extract text and real images from one chapter PDF."""
    doc     = fitz.open(ch["pdf_path"])
    img_dir = f"./images/{ch['subject']}_ch{ch['chapter_num']:02d}"
    os.makedirs(img_dir, exist_ok=True)
    pages   = []

    for page_num, page in enumerate(doc, 1):
        text      = clean_text(page.get_text())
        img_paths = []

        for img_info in page.get_images(full=True):
            xref, _, w, h = img_info[0], img_info[1], img_info[2], img_info[3]
            if not is_real_image(w, h): continue
            try:
                raw  = doc.extract_image(xref)
                dest = f"{img_dir}/p{page_num:03d}_{w}x{h}.{raw['ext']}"
                open(dest, "wb").write(raw["image"])
                img_paths.append(dest)
            except: pass

        pages.append({
            "page_num":    page_num,
            "text":        text,
            "image_paths": img_paths,
        })

    ch["pages"]        = pages
    ch["total_pages"]  = len(pages)
    ch["total_images"] = sum(len(p["image_paths"]) for p in pages)
    return ch


# Run extraction on all selected chapters
extracted = []
for ch in chapters_to_process:
    ch = extract_chapter(ch)
    extracted.append(ch)
    print(f"  ✅ {ch['subject']} Ch{ch['chapter_num']:02d} '{ch['chapter_title']}' "
          f"| Pages: {ch['total_pages']} | Images: {ch['total_images']}")

print(f"\n✅ Extracted {len(extracted)} chapters")

  ✅ Accountancy_Part1 Ch02 'Accounting for Partnership: Basic Concepts' | Pages: 59 | Images: 119
  ✅ Accountancy_Part1 Ch03 'Reconstitution of a Partnership Firm — Admission of a Partner' | Pages: 49 | Images: 99
  ✅ Accountancy_Part1 Ch04 'Reconstitution of a Partnership Firm — Retirement and Death' | Pages: 39 | Images: 79
  ✅ Accountancy_Part2 Ch01 'Accounting for Share Capital' | Pages: 74 | Images: 149
  ✅ Accountancy_Part2 Ch02 'Issue and Redemption of Debentures' | Pages: 69 | Images: 139
  ✅ Accountancy_Part2 Ch03 'Financial Statements of a Company' | Pages: 27 | Images: 55
  ✅ Accountancy_Part2 Ch04 'Analysis of Financial Statements' | Pages: 23 | Images: 47
  ✅ Accountancy_Part2 Ch05 'Accounting Ratios' | Pages: 47 | Images: 95
  ✅ Accountancy_Part2 Ch06 'Cash Flow Statement' | Pages: 46 | Images: 93
  ✅ Biology Ch01 'Sexual Reproduction in Flowering Plants' | Pages: 25 | Images: 109
  ✅ Biology Ch02 'Human Reproduction' | Pages: 15 | Images: 44
  ✅ Biology Ch03 'Reproductiv

### 👀 Preview — Text and Images from first chapter

In [11]:
from IPython.display import display, Image as IPImage

ch = extracted[0]
print(f"Subject  : {ch['subject']}")
print(f"Chapter  : {ch['chapter_title']}")
print(f"Pages    : {ch['total_pages']}")
print(f"Images   : {ch['total_images']}")
print(f"\n--- Page 1 text preview ---")
print(ch["pages"][0]["text"][:500], "...")

print("\n--- First 2 images ---")
shown = 0
for p in ch["pages"]:
    for img_path in p["image_paths"]:
        if shown >= 2: break
        print(f"Page {p['page_num']} | {os.path.getsize(img_path)//1024} KB")
        # display(IPImage(img_path, width=380))
        shown += 1
    if shown >= 2: break
if shown == 0:
    print("No images in first chapter")

Subject  : Accountancy_Part1
Chapter  : Accounting for Partnership: Basic Concepts
Pages    : 59
Images   : 119

--- Page 1 text preview ---
LEARNING OBJECTIVES After studying this chapter you will be able to: • Explain the concept of reconstitution of a partnership firm; • Identify the matters that need adjustments in the books of firm when a new partner is admitted; • Determine the new profit sharing ratio and calculate the sacrificing ratio; • Define goodwill and enumerate the factors that affect it; • Explain the methods of valuation of goodwill; • Describe how goodwill will be treated under different situations when a new partne ...

--- First 2 images ---
Page 1 | 2 KB
Page 1 | 8 KB


## ✂️ Step 6 — Chunk into RAG-Ready Pieces

In [12]:
def make_chunks(ch):
    all_words, word_pages = [], []
    for p in ch["pages"]:
        words = p["text"].split()
        all_words.extend(words)
        word_pages.extend([p["page_num"]] * len(words))

    chunks, i, t = [], 0, 0
    while i < len(all_words):
        window_words = all_words[i : i + CHUNK_SIZE]
        window_pages = word_pages[i : i + CHUNK_SIZE]
        text         = " ".join(window_words).strip()
        if len(text) > 80:
            unique_pages = sorted(set(window_pages))
            chunks.append({
                "id":   f"c12_{ch['subject']}_ch{ch['chapter_num']:02d}_t{t:04d}",
                "text": text,
                "metadata": {
                    "class":         "12",
                    "subject":       ch["subject"].replace("_", " "),
                    "chapter":       str(ch["chapter_num"]),
                    "chapter_title": ch["chapter_title"],
                    "page":          str(unique_pages[0]),
                    "pages_spanned": ",".join(str(p) for p in unique_pages),
                    "content_type":  "text",
                    "namespace":     ch["namespace"],
                },
            })
            t += 1
        i += CHUNK_SIZE - CHUNK_OVERLAP

    # ✅ Image chunks skipped for now — placeholders add no value
    # Will be replaced with real Qwen captions on RTX 5090

    return chunks


all_chunks = []
for ch in extracted:
    chunks = make_chunks(ch)
    all_chunks.extend(chunks)
    n_t = sum(1 for c in chunks if c["metadata"]["content_type"] == "text")
    n_i = sum(1 for c in chunks if c["metadata"]["content_type"] == "image_description")
    print(f"  {ch['subject']} Ch{ch['chapter_num']:02d} '{ch['chapter_title']}'"
          f" → {n_t} text + {n_i} image chunks")

print(f"\n✅ Total chunks: {len(all_chunks)}")

  Accountancy_Part1 Ch02 'Accounting for Partnership: Basic Concepts' → 22 text + 0 image chunks
  Accountancy_Part1 Ch03 'Reconstitution of a Partnership Firm — Admission of a Partner' → 18 text + 0 image chunks
  Accountancy_Part1 Ch04 'Reconstitution of a Partnership Firm — Retirement and Death' → 13 text + 0 image chunks
  Accountancy_Part2 Ch01 'Accounting for Share Capital' → 28 text + 0 image chunks
  Accountancy_Part2 Ch02 'Issue and Redemption of Debentures' → 23 text + 0 image chunks
  Accountancy_Part2 Ch03 'Financial Statements of a Company' → 9 text + 0 image chunks
  Accountancy_Part2 Ch04 'Analysis of Financial Statements' → 7 text + 0 image chunks
  Accountancy_Part2 Ch05 'Accounting Ratios' → 15 text + 0 image chunks
  Accountancy_Part2 Ch06 'Cash Flow Statement' → 14 text + 0 image chunks
  Biology Ch01 'Sexual Reproduction in Flowering Plants' → 10 text + 0 image chunks
  Biology Ch02 'Human Reproduction' → 6 text + 0 image chunks
  Biology Ch03 'Reproductive Health'

In [13]:
# # Wipe all namespaces
# ALL_NAMESPACES = [
#     "accountancy", "biology", "biotechnology", "business-studies",
#     "chemistry", "computer-science", "english", "informatics-practices",
#     "mathematics", "physics", "political-science", "psychology"
# ]

# for ns in ALL_NAMESPACES:
#     index.delete(delete_all=True, namespace=ns)
# print("🗑️  Cleared")


### 👀 Preview a Chunk

In [14]:
import json
sample = all_chunks[5]
print("ID      :", sample["id"])
print("Metadata:", json.dumps(sample["metadata"], indent=2))
print("\nText preview:")
print(sample["text"][:300], "...")

ID      : c12_Accountancy_Part1_ch02_t0005
Metadata: {
  "class": "12",
  "subject": "Accountancy Part1",
  "chapter": "2",
  "chapter_title": "Accounting for Partnership: Basic Concepts",
  "page": "14",
  "pages_spanned": "14,15,16,17",
  "content_type": "text",
  "namespace": "accountancy"
}

Text preview:
Annual salary to partners is Rs. 6,000 each. The profits for the last 3 years were Rs. 30,000; Rs. 36,000 and Rs. 42,000. Goodwill is to be valued at 2 years purchase of the last 3 years’ average super profits. Calculate the goodwill of the firm. Solution Interest on capital = 1,00,000 15 = Rs. 15, ...


## 🧠 Step 7 — Generate OpenAI Embeddings

In [15]:
from openai import OpenAI

oai_client = OpenAI(api_key=OPENAI_API_KEY)

def embed_chunks(chunks, batch_size=100):
    embedded = []
    for i in range(0, len(chunks), batch_size):
        batch = chunks[i : i + batch_size]
        resp  = oai_client.embeddings.create(
            input=[c["text"] for c in batch],
            model=EMBED_MODEL,
        )
        for j, obj in enumerate(resp.data):
            c = batch[j].copy()
            c["embedding"] = obj.embedding
            embedded.append(c)
        print(f"  Batch {i//batch_size + 1}: {len(batch)} chunks embedded")
    return embedded

print(f"Embedding {len(all_chunks)} chunks...")
embedded_chunks = embed_chunks(all_chunks)
print(f"\n✅ Done! Dimension: {len(embedded_chunks[0]['embedding'])}")

Embedding 1309 chunks...
  Batch 1: 100 chunks embedded
  Batch 2: 100 chunks embedded
  Batch 3: 100 chunks embedded
  Batch 4: 100 chunks embedded
  Batch 5: 100 chunks embedded
  Batch 6: 100 chunks embedded
  Batch 7: 100 chunks embedded
  Batch 8: 100 chunks embedded
  Batch 9: 100 chunks embedded
  Batch 10: 100 chunks embedded
  Batch 11: 100 chunks embedded
  Batch 12: 100 chunks embedded
  Batch 13: 100 chunks embedded
  Batch 14: 9 chunks embedded

✅ Done! Dimension: 1536


## 📦 Step 8 — Upload to Pinecone

In [16]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=PINECONE_API_KEY)

existing = [i.name for i in pc.list_indexes()]
if PINECONE_INDEX not in existing:
    print(f"Creating index '{PINECONE_INDEX}'...")
    pc.create_index(
        name=PINECONE_INDEX,
        dimension=1536,
        metric="cosine",
        spec=ServerlessSpec(cloud=PINECONE_CLOUD, region=PINECONE_REGION),
    )
    print("  ✅ Index created")
else:
    print(f"  ✅ Index '{PINECONE_INDEX}' already exists")

index = pc.Index(PINECONE_INDEX)
stats = index.describe_index_stats()
print(f"\n📊 Current index stats:")
print(f"   Total vectors : {stats.total_vector_count}")
print(f"   Namespaces    : {list(stats.namespaces.keys())}")

Creating index 'cbse-class12-from-pc2'...
  ✅ Index created

📊 Current index stats:
   Total vectors : 0
   Namespaces    : []


In [17]:
len(embedded_chunks)

1309

In [18]:
import time

def upsert_to_pinecone(chunks, batch_size=100):
    by_ns = {}
    for c in chunks:
        by_ns.setdefault(c["metadata"]["namespace"], []).append(c)

    for ns, items in by_ns.items():
        vectors = [
            {"id": c["id"], "values": c["embedding"],
             "metadata": {**c["metadata"], "text": c["text"]}}
            for c in items
        ]
        for i in range(0, len(vectors), batch_size):
            index.upsert(vectors=vectors[i : i + batch_size], namespace=ns)
        print(f"  ✅ {len(items):>5} vectors → [{ns}]")

print("Upserting to Pinecone...")
upsert_to_pinecone(embedded_chunks)

time.sleep(3)
stats = index.describe_index_stats()
print(f"\n📊 Final index stats:")
print(f"   Total vectors : {stats.total_vector_count}")
for ns, info in stats.namespaces.items():
    print(f"   [{ns}] → {info.vector_count} vectors")

Upserting to Pinecone...
  ✅   149 vectors → [accountancy]
  ✅   100 vectors → [biology]
  ✅   117 vectors → [biotechnology]
  ✅   132 vectors → [business-studies]
  ✅   118 vectors → [chemistry]
  ✅    92 vectors → [computer-science]
  ✅    44 vectors → [english]
  ✅    67 vectors → [informatics-practices]
  ✅   127 vectors → [mathematics]
  ✅   167 vectors → [physics]
  ✅   117 vectors → [political-science]
  ✅    79 vectors → [psychology]

📊 Final index stats:
   Total vectors : 1309
   [accountancy] → 149 vectors
   [psychology] → 79 vectors
   [physics] → 167 vectors
   [english] → 44 vectors
   [mathematics] → 127 vectors
   [business-studies] → 132 vectors
   [political-science] → 117 vectors
   [computer-science] → 92 vectors
   [informatics-practices] → 67 vectors
   [chemistry] → 118 vectors
   [biotechnology] → 117 vectors
   [biology] → 100 vectors


In [19]:
import time

stats = index.describe_index_stats()
print(f"Total vectors : {stats.total_vector_count}")
for ns, info in stats.namespaces.items():
    print(f"  [{ns}] → {info.vector_count} vectors")

Total vectors : 1309
  [chemistry] → 118 vectors
  [computer-science] → 92 vectors
  [business-studies] → 132 vectors
  [biotechnology] → 117 vectors
  [informatics-practices] → 67 vectors
  [biology] → 100 vectors
  [physics] → 167 vectors
  [political-science] → 117 vectors
  [english] → 44 vectors
  [mathematics] → 127 vectors
  [accountancy] → 149 vectors
  [psychology] → 79 vectors


## 🔍 Step 9 — Test Retrieval

**Note:** `namespace=None` searches Pinecone's empty default namespace (0 vectors).
For "All Subjects", we query each namespace separately and merge by score.

In [20]:
# All namespaces that were ingested
ALL_NAMESPACES = list(set(c["namespace"] for c in chapters_to_process))

SUBJECT_NAMESPACE_MAP = {
    "All Subjects":          None,
    "Biology":               "biology",
    "Biotechnology":         "biotechnology",
    "Chemistry":             "chemistry",
    "Physics":               "physics",
    "Mathematics":           "mathematics",
    "Computer Science":      "computer-science",
    "Informatics Practices": "informatics-practices",
    "English":               "english",
    "Accountancy":           "accountancy",
    "Business Studies":      "business-studies",
    "Political Science":     "political-science",
    "Psychology":            "psychology",
}

# Filter out image placeholders from search results
# (they have no real content until vision AI runs)
def retrieve(query, namespace=None, top_k=TOP_K, filters=None):
    vec = oai_client.embeddings.create(
        input=[query], model=EMBED_MODEL
    ).data[0].embedding

    # ✅ Always exclude image placeholders from results
    base_filter = {"content_type": {"$eq": "text"}}

    # Merge with any page/chapter filters
    if filters:
        final_filter = {"$and": [base_filter, filters]}
    else:
        final_filter = base_filter

    kwargs = {
        "vector":           vec,
        "top_k":            top_k,
        "include_metadata": True,
        "filter":           final_filter,
    }

    if namespace:
        kwargs["namespace"] = namespace
        return index.query(**kwargs).matches
    else:
        all_matches = []
        for ns in ALL_NAMESPACES:
            kwargs["namespace"] = ns
            all_matches.extend(index.query(**kwargs).matches)
        return sorted(all_matches, key=lambda x: x.score, reverse=True)[:top_k]

def parse_filters(question):
    filters = {}
    ch_match = re.search(r"chapter\s*(\d+)", question.lower())
    if ch_match:
        filters["chapter"] = ch_match.group(1)

    pg_match = re.search(r"(?:page|pg)\.?\s*(\d+)", question.lower())
    if pg_match:
        filters["pages_spanned"] = {"$in": [pg_match.group(1)]}

    return filters if filters else None


# Test queries — adjust based on which subjects you ingested
test_queries = [
    ("What is fertilisation in flowering plants?", "biology"),
    ("What is electric flux?",                    "physics"),
    ("What is the solid state?",                  "chemistry"),
    ("Explain recursion in Python",               None),       # All Subjects
]

for query, ns in test_queries:
    label = f"[{ns}]" if ns else "[All Subjects]"
    print(f"\n🔍 {label} '{query}'")
    matches = retrieve(query, namespace=ns)
    if matches:
        m = matches[0]
        print(f"   Score   : {m.score:.3f}")
        print(f"   Source  : {m.metadata.get('subject')} | "
              f"{m.metadata.get('chapter_title')} | Page {m.metadata.get('page')}")
        print(f"   Preview : {m.metadata.get('text','')[:100]}...")
    else:
        print("   No results found")


🔍 [biology] 'What is fertilisation in flowering plants?'
   Score   : 0.835
   Source  : Biology | Sexual Reproduction in Flowering Plants | Page 22
   Preview : seeds can remain alive for hundreds of years. There are several records of very old yet viable seeds...

🔍 [physics] 'What is electric flux?'
   Score   : 0.856
   Source  : Physics Part1 | Electric Charges and Fields | Page 21
   Preview : Therefore, the flux going out of the surface dS is v. ˆn dS. For the case of the electric field, we ...

🔍 [chemistry] 'What is the solid state?'
   Score   : 0.770
   Source  : Chemistry Part1 | The Solid State | Page 1
   Preview : After studying this Unit, you will be able to describe the formation of different types of solutions...

🔍 [All Subjects] 'Explain recursion in Python'
   Score   : 0.778
   Source  : Computer Science | Stack | Page 1
   Preview : In this Chapter Introduction Stack Operations on Stack Implementation of Stack in Python Notations f...


## 🤖 Step 10 — RAG Answer Generation

In [21]:
SYSTEM_PROMPT = """You are a dedicated CBSE Class XII study assistant.
Your knowledge comes ONLY from official NCERT Class XII textbooks.

Rules:
- Answer using ONLY the retrieved textbook content provided
- Always cite: subject, chapter name, and page number
- Use clear, exam-friendly language for a Class XII student
- If the answer is not in the retrieved content, say so honestly
- Never use knowledge outside the provided context"""


def format_context(matches):
    if not matches: return "No relevant content found."
    parts = []
    for i, m in enumerate(matches, 1):
        meta  = m.metadata
        icon  = "📊" if meta.get("content_type") == "image_description" else "📖"
        label = (f"[{i}] {icon} {meta.get('subject')} | "
                 f"{meta.get('chapter_title')} | Page {meta.get('page')}")
        parts.append(f"{label}\n{meta.get('text', '')}")
    return "\n\n---\n\n".join(parts)


def format_sources(matches):
    seen, lines = set(), []
    for m in matches:
        meta = m.metadata
        icon = "📊" if meta.get("content_type") == "image_description" else "📖"
        key  = (f"{icon} {meta.get('subject')} | "
                f"{meta.get('chapter_title')} | Page {meta.get('page')}")
        if key not in seen:
            seen.add(key)
            lines.append(key)
    return "**Sources:**\n" + "\n".join(lines) if lines else ""


def ask(question, subject_filter="All Subjects"):
    ns      = SUBJECT_NAMESPACE_MAP.get(subject_filter)
    filters = parse_filters(question)
    if filters:
        print(f"  🔎 Filter: {filters}")
    matches = retrieve(question, namespace=ns, filters=filters)
    context = format_context(matches)
    sources = format_sources(matches)

    resp = oai_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": f"Question: {question}\n\nContext:\n{context}"},
        ],
        temperature=0.1,
        max_tokens=700,
    )
    answer = resp.choices[0].message.content.strip()
    return f"{answer}\n\n---\n{sources}" if sources else answer


print("✅ ask() ready")

✅ ask() ready


In [22]:
# Run test questions — adjust subjects based on what you ingested

test_qs = [
    ("What is the process of fertilisation in flowering plants?", "Biology"),
    ("What is Coulomb's law?",                                   "Physics"),
    ("Explain the difference between crystalline and amorphous solids", "Chemistry"),
    ("What is recursion in Python?",                             "Computer Science"),
    ("What is meiosis?",                                         "All Subjects"),
]

for q, subj in test_qs:
    print(f"\n{'='*60}")
    print(f"❓ [{subj}] {q}")
    print(f"{'='*60}")
    print(ask(q, subj))


❓ [Biology] What is the process of fertilisation in flowering plants?
The process of fertilisation in flowering plants involves a unique event called double fertilisation. This process occurs within the embryo sac of the ovule. Here is a detailed explanation:

1. **Pollen Tube Entry**: After pollination, the pollen grain germinates on the stigma, and a pollen tube grows through the style to reach the ovule. The pollen tube enters one of the synergids in the embryo sac.

2. **Release of Male Gametes**: The pollen tube releases two male gametes into the cytoplasm of the synergid.

3. **Syngamy**: One of the male gametes moves towards the egg cell and fuses with its nucleus. This fusion is called syngamy, resulting in the formation of a diploid zygote.

4. **Triple Fusion**: The other male gamete fuses with the two polar nuclei located in the central cell of the embryo sac. This fusion involves three haploid nuclei and is termed triple fusion, forming a triploid primary endosperm nucleus

## 🚀 Step 11 — Gradio Chatbot UI
> A public URL appears below — share or open in any browser.

In [ ]:
# Option 1 — Page specific
print(ask("What is on page 6 and 7 of Queue chapter?", "Computer Science"))

# Option 2 — Topic specific
print(ask("What is covered in the Queue chapter pages 6 and 7?", "Computer Science"))

# Option 3 — Natural question
print(ask("Explain the content from page 6 of Queue in Computer Science", "Computer Science"))


  🔎 Filter: {'pages_spanned': {'$in': ['6']}}
I'm sorry, but I don't have access to the specific content of page 6 and 7 of the Queue chapter from the NCERT Class XII textbooks. If you have any other questions or need information from a different section, feel free to ask!
The content on pages 6 and 7 of the Queue chapter covers the following:

1. **Python Implementation of Queue**: 
   - A program example is provided to simulate a queue at a bank cash counter using Python. The program demonstrates how to enqueue and dequeue elements, check the size of the queue, and handle underflow situations when the queue is empty.
   - The program uses a list to represent the queue and defines functions for enqueueing (`enqueue`), dequeueing (`dequeue`), checking if the queue is empty (`isEmpty`), and getting the size of the queue (`size`).

2. **Activities**:
   - Activity 4.1 asks how to avoid printing `None` when trying to print an empty queue.
   - Activity 4.2 asks for a function to list the 

In [69]:
# import gradio as gr

# INGESTED_SUBJECTS = ["All Subjects"] + sorted(set(
#     k for k, v in SUBJECT_NAMESPACE_MAP.items()
#     if v in ALL_NAMESPACES and k != "All Subjects"
# ))

# SAMPLE_QS = "\n".join([
#     "### 💡 Try asking:",
#     "- What is meiosis?",
#     "- How does fertilisation occur in plants?",
#     "- What is Coulomb's law?",
#     "- Explain Gauss's law",
#     "- What is the elevation of boiling point?",
#     "- Difference between crystalline and amorphous solids",
#     "- What is a stack in Python?",
#     "- Explain the principles of management",
# ])

# def chat_fn(question, subject, history):
#     if not question.strip(): return history, ""
#     history.append({"role": "user",      "content": question})
#     history.append({"role": "assistant", "content": ask(question, subject)})
#     return history, ""

# with gr.Blocks(title="CBSE Class XII Learning Buddy", theme=gr.themes.Soft()) as demo:
#     gr.HTML("""
#     <div style="text-align:center; padding:16px 0 8px">
#         <h1>📚 CBSE Class XII Learning Buddy</h1>
#         <p style="color:#666">Powered by official NCERT Class XII textbooks</p>
#     </div>""")

#     with gr.Row():
#         with gr.Column(scale=4):
#             chatbot = gr.Chatbot(height=480, type="messages")
#             with gr.Row():
#                 q_box   = gr.Textbox(
#                     placeholder="Ask anything from your Class XII textbook...",
#                     label="", scale=5, container=False
#                 )
#                 ask_btn = gr.Button("Ask →", variant="primary", scale=1)

#         with gr.Column(scale=1, min_width=210):
#             gr.Markdown("### ⚙️ Options")
#             subj_dd   = gr.Dropdown(
#                 choices=INGESTED_SUBJECTS,
#                 value="All Subjects",
#                 label="Filter by Subject",
#             )
#             clear_btn = gr.Button("🗑️ Clear Chat", variant="secondary")
#             gr.Markdown("---")
#             gr.Markdown(SAMPLE_QS)

#     history_state = gr.State([])
#     ask_btn.click(chat_fn, inputs=[q_box, subj_dd, history_state], outputs=[chatbot, q_box])
#     q_box.submit(chat_fn,  inputs=[q_box, subj_dd, history_state], outputs=[chatbot, q_box])
#     clear_btn.click(lambda: ([], []), outputs=[chatbot, history_state])

# demo.launch(share=True, quiet=True)

In [70]:
# # Run this before relaunching
# try:
#     demo.close()
# except:
#     pass

---
## 🧹 Cleanup — Delete Pinecone Index (Optional)

In [ ]:
# Uncomment to wipe the index
# pc.delete_index(PINECONE_INDEX)
# print(f"🗑️  Deleted: {PINECONE_INDEX}")
# print("Uncomment above to delete the index")

## Deploying to huggingface
---

In [34]:
# Quick checklist before deploying:
print("PINECONE_INDEX :", PINECONE_INDEX)   # cbse-class12
print("OPENAI_API_KEY :", OPENAI_API_KEY[:8], "...")
print("PINECONE_API_KEY:", PINECONE_API_KEY[:8], "...")

PINECONE_INDEX : cbse-class12-from-pc2
OPENAI_API_KEY : sk-proj- ...
PINECONE_API_KEY: pcsk_42W ...


In [36]:
!uv pip install huggingface_hub

Audited 1 package in 2ms


In [37]:
from huggingface_hub import login
login()

In [38]:
from huggingface_hub import HfApi

api      = HfApi()
username = api.whoami()["name"]
repo_id  = f"{username}/cbse-learning-buddy"

api.create_repo(repo_id=repo_id, repo_type="space", space_sdk="gradio", exist_ok=True)
print(f"✅ https://huggingface.co/spaces/{repo_id}")

✅ https://huggingface.co/spaces/balaprasannav2009/cbse-learning-buddy


In [40]:
with open("requirements.txt", "w") as f:
    f.write("openai>=1.30.0\npinecone>=3.0.0\ngradio\n")
print("✅ requirements.txt created")

✅ requirements.txt created


In [42]:
api.add_space_secret(repo_id=repo_id, key="OPENAI_API_KEY",   value=OPENAI_API_KEY)
api.add_space_secret(repo_id=repo_id, key="PINECONE_API_KEY", value=PINECONE_API_KEY)
api.add_space_secret(repo_id=repo_id, key="PINECONE_INDEX",   value="cbse-class12-from-pc2")
print("✅ Secrets added — Space is building!")
print(f"🌐 https://huggingface.co/spaces/{repo_id}")

✅ Secrets added — Space is building!
🌐 https://huggingface.co/spaces/balaprasannav2009/cbse-learning-buddy


In [41]:
api.upload_file(path_or_fileobj="app.py",           path_in_repo="app.py",           repo_id=repo_id, repo_type="space")
api.upload_file(path_or_fileobj="requirements.txt", path_in_repo="requirements.txt", repo_id=repo_id, repo_type="space")
print("✅ Files uploaded")

✅ Files uploaded


In [62]:
# api.delete_repo(repo_id=repo_id, repo_type="space")
# print("🗑️ Deleted old space")

🗑️ Deleted old space
